In [21]:
from pathlib import Path

video_dir = Path("trimmed_woo_pitch_videos")
file_names = sorted([p.name for p in video_dir.iterdir() if p.is_file()])

print(f"Found {len(file_names)} files in '{video_dir}'.")
file_names[:10]

Found 5912 files in 'trimmed_woo_pitch_videos'.


['PitchType-CH_Zone-11_PlayID-052f3798-af8f-3cd2-b07d-ea321494c47b_Date-2025-06-17.mp4',
 'PitchType-CH_Zone-11_PlayID-060b0c26-9caf-43d8-b811-2a94bed53c0f_Date-2023-09-18.mp4',
 'PitchType-CH_Zone-11_PlayID-0be2c535-bfab-42d2-a8d4-358e30c67e3a_Date-2024-08-02.mp4',
 'PitchType-CH_Zone-11_PlayID-0e83ceb2-477d-3e53-b6ae-f72acdba6fdf_Date-2025-04-06.mp4',
 'PitchType-CH_Zone-11_PlayID-18fc5860-090b-4a3e-b753-fea9fb841a76_Date-2023-09-18.mp4',
 'PitchType-CH_Zone-11_PlayID-1d59a276-26ce-4136-8999-62d0c40b6d05_Date-2024-09-17.mp4',
 'PitchType-CH_Zone-11_PlayID-2b2d8fd4-c6dc-4c2e-9092-88849e2967d0_Date-2024-05-10.mp4',
 'PitchType-CH_Zone-11_PlayID-3a2074ae-a169-33e9-a00b-ea39c0987728_Date-2025-06-11.mp4',
 'PitchType-CH_Zone-11_PlayID-443565ce-2d2b-32fb-8f71-c778af0e4391_Date-2025-06-28.mp4',
 'PitchType-CH_Zone-11_PlayID-4c9dda04-404a-4960-9179-603a35accf70_Date-2023-08-28.mp4']

In [22]:
def get_pitch_type(file_name):
    """Extract the pitch type from a file name."""
    return file_name.split("_")[0].split("-")[1]

In [23]:
get_pitch_type(file_names[0])

'CH'

In [24]:
VALID_PITCH_TYPES = {"CH", "CS", "CU", "EP", "FA", "FC", "FF", "FO", "FS", "IN", "KC", "KN", "PO", "SC", "SI", "SL", "ST", "SV"}

def is_valid_pitch_type(file_name):
    """Verify that the pitch type extracted from a file name is valid."""
    pitch_type = get_pitch_type(file_name)
    return pitch_type in VALID_PITCH_TYPES

# Verify all files have valid pitch types
invalid_files = [f for f in file_names if not is_valid_pitch_type(f)]
print(f"Files with invalid pitch types: {len(invalid_files)}")
if invalid_files:
    for f in invalid_files[:10]:
        print(f"  {f} -> {get_pitch_type(f)}")
else:
    print("All files have valid pitch types.")

Files with invalid pitch types: 0
All files have valid pitch types.


In [25]:
def get_play_id(file_name):
    """Extract the play ID from a file name."""
    return file_name.split("PlayID-")[1].split("_")[0]

In [ ]:
VALID_PITCH_TYPES = {"CH", "CS", "CU", "EP", "FA", "FC", "FF", "FO", "FS", "IN", "KC", "KN", "PO", "SC", "SI", "SL", "ST", "SV"}

In [50]:
video_pitch_types = set(get_pitch_type(f) for f in file_names)
savant_pitch_types = set(savant_df["pitch_type"].dropna().unique())

print(f"Pitch types in video files: {sorted(video_pitch_types)}")
print(f"Pitch types in savant data: {sorted(savant_pitch_types)}")

missing_from_videos = VALID_PITCH_TYPES - video_pitch_types
missing_from_savant = VALID_PITCH_TYPES - savant_pitch_types

print(f"\nValid pitch types missing from video files: {sorted(missing_from_videos) if missing_from_videos else 'None'}")
print(f"Valid pitch types missing from savant data: {sorted(missing_from_savant) if missing_from_savant else 'None'}")

# Also check if there are types in the data that aren't in VALID_PITCH_TYPES
extra_in_videos = video_pitch_types - VALID_PITCH_TYPES
extra_in_savant = savant_pitch_types - VALID_PITCH_TYPES
if extra_in_videos:
    print(f"\nVideo pitch types not in VALID_PITCH_TYPES: {sorted(extra_in_videos)}")
if extra_in_savant:
    print(f"Savant pitch types not in VALID_PITCH_TYPES: {sorted(extra_in_savant)}")

Pitch types in video files: ['CH', 'FF', 'SI', 'SL', 'ST']
Pitch types in savant data: ['CH', 'FF', 'SI', 'SL', 'ST']

Valid pitch types missing from video files: ['CS', 'CU', 'EP', 'FA', 'FC', 'FO', 'FS', 'IN', 'KC', 'KN', 'PO', 'SC', 'SV']
Valid pitch types missing from savant data: ['CS', 'CU', 'EP', 'FA', 'FC', 'FO', 'FS', 'IN', 'KC', 'KN', 'PO', 'SC', 'SV']


In [26]:
def get_game_date(file_name):
    """Extract the game date from a file name."""
    return file_name.split("Date-")[1].split(".")[0]

In [27]:
get_game_date(file_names[0])

'2025-06-17'

In [28]:
from collections import defaultdict

def get_files_by_date(file_names):
    """Group file names by their game date and return a dict of date -> list of file names."""
    files_by_date = defaultdict(list)
    for f in file_names:
        date = get_game_date(f)
        files_by_date[date].append(f)
    return dict(files_by_date)

def get_savant_df_by_date(savant_df, date):
    """Return a subset of the savant DataFrame for a specific game date (YYYY-MM-DD)."""
    return savant_df[savant_df["game_date"] == date].reset_index(drop=True)

# Example usage
files_by_date = get_files_by_date(file_names)
print(f"Found files across {len(files_by_date)} unique game dates.")
for date in sorted(files_by_date.keys())[:103]:
    print(f"  {date}: {len(files_by_date[date])} files")

Found files across 102 unique game dates.
  2023-06-03: 37 files
  2023-06-10: 80 files
  2023-06-16: 89 files
  2023-06-22: 93 files
  2023-06-27: 87 files
  2023-07-03: 86 files
  2023-07-08: 78 files
  2023-07-18: 83 files
  2023-07-23: 88 files
  2023-07-29: 72 files
  2023-08-03: 83 files
  2023-08-22: 65 files
  2023-08-28: 69 files
  2023-09-04: 81 files
  2023-09-12: 83 files
  2023-09-18: 86 files
  2023-09-24: 77 files
  2023-09-29: 82 files
  2024-05-10: 62 files
  2024-05-15: 79 files
  2024-05-21: 77 files
  2024-05-26: 69 files
  2024-05-31: 66 files
  2024-06-06: 85 files
  2024-06-19: 63 files
  2024-06-24: 60 files
  2024-07-12: 65 files
  2024-07-21: 72 files
  2024-07-27: 70 files
  2024-08-02: 68 files
  2024-08-08: 88 files
  2024-08-14: 85 files
  2024-08-19: 86 files
  2024-08-25: 92 files
  2024-08-31: 65 files
  2024-09-05: 87 files
  2024-09-11: 88 files
  2024-09-17: 92 files
  2024-09-22: 81 files
  2024-09-27: 76 files
  2025-03-30: 74 files
  2025-04-06: 8

In [33]:
from collections import Counter

years = [get_game_date(f)[:4] for f in file_names]
files_per_year = Counter(years)

for year in sorted(files_per_year):
    print(f"  {year}: {files_per_year[year]} files")

  2023: 1419 files
  2024: 1676 files
  2025: 2744 files
  Matc: 73 files


In [34]:
from collections import defaultdict

def get_files_by_year(file_names):
    """Group file names by their game year and return a dict of year -> list of file names."""
    files_by_year = defaultdict(list)
    for f in file_names:
        year = get_game_date(f)[:4]
        files_by_year[year].append(f)
    return dict(files_by_year)

files_by_year = get_files_by_year(file_names)
print(f"Found files across {len(files_by_year)} unique years.")
for year in sorted(files_by_year.keys()):
    print(f"  {year}: {len(files_by_year[year])} files")

Found files across 4 unique years.
  2023: 1419 files
  2024: 1676 files
  2025: 2744 files
  Matc: 73 files


In [35]:
files_by_year.keys()

dict_keys(['2025', '2023', '2024', 'Matc'])

In [37]:
len(files_by_year['2023'])

1419

In [38]:
len(files_by_year['2024'])

1676

In [39]:
len(files_by_year['2025'])

2744

In [45]:
# Filter to only 2025 files, then get the n most recent games
recent_files_2025, recent_dates_2025 = get_n_most_recent_games(files_by_year['2025'], 8)
print(f"Most recent {len(recent_dates_2025)} game dates in 2025:")
for date in sorted(recent_dates_2025):
    print(f"  {date}: {len(get_files_by_date(files_by_year['2025'])[date])} files")
print(f"\nTotal files from recent 2025 games: {len(recent_files_2025)}")

Most recent 8 game dates in 2025:
  2025-08-22: 87 files
  2025-08-27: 94 files
  2025-09-02: 83 files
  2025-09-08: 94 files
  2025-09-13: 99 files
  2025-09-19: 65 files
  2025-10-17: 28 files
  2025-10-20: 38 files

Total files from recent 2025 games: 588


In [46]:
def get_n_most_recent_games(file_list, n):
    """Return files and dates from the n most recent games in the given file list."""
    files_grouped = get_files_by_date(file_list)
    # Sort dates in descending order (most recent first)
    sorted_dates = sorted(files_grouped.keys(), reverse=True)[:n]
    recent_files = []
    for date in sorted_dates:
        recent_files.extend(files_grouped[date])
    return recent_files, sorted_dates

# Get the 16 most recent games across all years (excluding 'Matc' keys)
all_dated_files = files_by_year.get('2025', []) + files_by_year.get('2024', []) + files_by_year.get('2023', [])
recent_files, recent_dates = get_n_most_recent_games(all_dated_files, 16)
print(f"Most recent {len(recent_dates)} game dates:")
for date in sorted(recent_dates):
    print(f"  {date}: {len(get_files_by_date(all_dated_files)[date])} files")
print(f"\nTotal files from recent 16 games: {len(recent_files)}")

Most recent 16 game dates:
  2025-07-10: 103 files
  2025-07-15: 8 files
  2025-07-20: 88 files
  2025-07-25: 96 files
  2025-07-30: 80 files
  2025-08-05: 82 files
  2025-08-10: 96 files
  2025-08-16: 101 files
  2025-08-22: 87 files
  2025-08-27: 94 files
  2025-09-02: 83 files
  2025-09-08: 94 files
  2025-09-13: 99 files
  2025-09-19: 65 files
  2025-10-17: 28 files
  2025-10-20: 38 files

Total files from recent 16 games: 1242


In [47]:
# Organize recent files into a list of lists, one per game date (sorted most recent first)
recent_files_by_game = []
files_grouped = get_files_by_date(recent_files)
for date in sorted(files_grouped.keys(), reverse=True):
    recent_files_by_game.append(files_grouped[date])

print(f"Number of games: {len(recent_files_by_game)}")
for i, game_files in enumerate(recent_files_by_game):
    date = get_game_date(game_files[0])
    print(f"  Game {i+1} ({date}): {len(game_files)} files")

Number of games: 16
  Game 1 (2025-10-20): 38 files
  Game 2 (2025-10-17): 28 files
  Game 3 (2025-09-19): 65 files
  Game 4 (2025-09-13): 99 files
  Game 5 (2025-09-08): 94 files
  Game 6 (2025-09-02): 83 files
  Game 7 (2025-08-27): 94 files
  Game 8 (2025-08-22): 87 files
  Game 9 (2025-08-16): 101 files
  Game 10 (2025-08-10): 96 files
  Game 11 (2025-08-05): 82 files
  Game 12 (2025-07-30): 80 files
  Game 13 (2025-07-25): 96 files
  Game 14 (2025-07-20): 88 files
  Game 15 (2025-07-15): 8 files
  Game 16 (2025-07-10): 103 files


In [48]:
# Get file names for each file in the 16 most recent games, organized by game date
for i, game_files in enumerate(recent_files_by_game):
    date = get_game_date(game_files[0])
    print(f"\n--- Game {i+1} ({date}) ---")
    for f in sorted(game_files):
        print(f"  {f}")


--- Game 1 (2025-10-20) ---
  PitchType-FF_Zone-11_PlayID-2f2261f5-7382-33d4-a7d6-2461dd97d3b6_Date-2025-10-20.mp4
  PitchType-FF_Zone-11_PlayID-aeec2aa6-00e0-3de6-8753-cab844cf0919_Date-2025-10-20.mp4
  PitchType-FF_Zone-12_PlayID-1a347758-d20b-3f01-bd16-39c6a182ca65_Date-2025-10-20.mp4
  PitchType-FF_Zone-12_PlayID-5164cad8-3dac-3a8f-afcb-b8c41cb90481_Date-2025-10-20.mp4
  PitchType-FF_Zone-12_PlayID-81478a38-0b31-3ba2-9a86-76184f1f7e84_Date-2025-10-20.mp4
  PitchType-FF_Zone-14_PlayID-508c1058-2172-312a-909c-f18d7c44875a_Date-2025-10-20.mp4
  PitchType-FF_Zone-14_PlayID-570e3f56-7332-3e3c-ac9f-b68a2ec91582_Date-2025-10-20.mp4
  PitchType-FF_Zone-14_PlayID-67af52bc-01a5-3755-9c4d-92bd08e082cd_Date-2025-10-20.mp4
  PitchType-FF_Zone-14_PlayID-8d65fb43-4d5b-3cda-9b60-87914b6d16df_Date-2025-10-20.mp4
  PitchType-FF_Zone-14_PlayID-a78647cd-813a-3cc8-8d07-bbeb4589bd77_Date-2025-10-20.mp4
  PitchType-FF_Zone-14_PlayID-aa8d4469-235b-3857-b69a-169bfd363410_Date-2025-10-20.mp4
  PitchType-FF

In [ ]:
# Get file names for each file in the 16 most recent games, organized by game date
for i, game_files in enumerate(recent_files_by_game):
    date = get_game_date(game_files[0])
    print(f"\n--- Game {i+1} ({date}) ---")
    for f in sorted(game_files):
        print(f"  {f}")

In [51]:
import shutil
import random

In [81]:
def copy_files_to_classification_folders(file_names, base_dir="modules/naive_classification/test", video_dir=video_dir):
    """Copy video files into classification folders organized by pitch type."""
    pitch_type_labels = {
        "FF": "FF - Fastball",
        "SI": "SI - Sinker",
        "SL": "SL - Slider",
        "ST": "ST - Sweeper",
        "CH": "CH - Changeup",
    }

    base_path = Path(base_dir)

    copied = 0
    skipped = 0
    for f in file_names:
        pitch_type = get_pitch_type(f)
        if pitch_type in pitch_type_labels:
            dest_folder = base_path / pitch_type_labels[pitch_type]
            src = video_dir / f
            dest = dest_folder / f
            if not dest.exists():
                shutil.copy2(src, dest)
                copied += 1
            else:
                skipped += 1
        else:
            skipped += 1

    print(f"Copied {copied} files, skipped {skipped} files.")
    for folder_name in sorted(pitch_type_labels.values()):
        folder_path = base_path / folder_name
        count = len(list(folder_path.iterdir())) if folder_path.exists() else 0
        print(f"  {folder_name}: {count} files")

In [55]:
recent_files_by_game[1]

['PitchType-FF_Zone-11_PlayID-2af31aba-574d-3523-86e0-0b1638f911d7_Date-2025-10-17.mp4',
 'PitchType-FF_Zone-11_PlayID-2faf578e-26e9-3494-a3d9-db6944850aa1_Date-2025-10-17.mp4',
 'PitchType-FF_Zone-11_PlayID-5813ec6c-c2be-3b9e-8b59-720f2c682082_Date-2025-10-17.mp4',
 'PitchType-FF_Zone-12_PlayID-fb14c9cd-d45a-33a5-ad6a-f92facc8f03b_Date-2025-10-17.mp4',
 'PitchType-FF_Zone-1_PlayID-0453438d-aadc-39a9-bcb4-10977a6afa44_Date-2025-10-17.mp4',
 'PitchType-FF_Zone-1_PlayID-3abdc4f3-d956-3989-9c63-bfeb89536513_Date-2025-10-17.mp4',
 'PitchType-FF_Zone-1_PlayID-e8e5321b-bf93-3fe0-9302-ffa7cf2a332c_Date-2025-10-17.mp4',
 'PitchType-FF_Zone-2_PlayID-a40de416-dba9-3021-9a26-d3d4cf311c01_Date-2025-10-17.mp4',
 'PitchType-FF_Zone-2_PlayID-b3370c98-47d5-33ae-b3da-72a32e6df3f0_Date-2025-10-17.mp4',
 'PitchType-FF_Zone-5_PlayID-0b23c779-689b-3415-a645-55b325ce3358_Date-2025-10-17.mp4',
 'PitchType-FF_Zone-5_PlayID-1939f278-b028-3c15-9dc5-721939b72ad7_Date-2025-10-17.mp4',
 'PitchType-FF_Zone-5_PlayID

In [82]:
# Get file names for each file in the 8 most recent games, organized by game date
for i in range(0,8):
    copy_files_to_classification_folders(recent_files_by_game[i], base_dir="modules/naive_classification/test", video_dir = Path("trimmed_woo_pitch_videos"))

Copied 38 files, skipped 0 files.
  CH - Changeup: 0 files
  FF - Fastball: 22 files
  SI - Sinker: 11 files
  SL - Slider: 0 files
  ST - Sweeper: 5 files
Copied 28 files, skipped 0 files.
  CH - Changeup: 0 files
  FF - Fastball: 38 files
  SI - Sinker: 19 files
  SL - Slider: 0 files
  ST - Sweeper: 9 files
Copied 65 files, skipped 0 files.
  CH - Changeup: 1 files
  FF - Fastball: 62 files
  SI - Sinker: 39 files
  SL - Slider: 6 files
  ST - Sweeper: 23 files
Copied 99 files, skipped 0 files.
  CH - Changeup: 2 files
  FF - Fastball: 114 files
  SI - Sinker: 53 files
  SL - Slider: 17 files
  ST - Sweeper: 44 files
Copied 94 files, skipped 0 files.
  CH - Changeup: 10 files
  FF - Fastball: 159 files
  SI - Sinker: 80 files
  SL - Slider: 24 files
  ST - Sweeper: 51 files
Copied 83 files, skipped 0 files.
  CH - Changeup: 13 files
  FF - Fastball: 203 files
  SI - Sinker: 98 files
  SL - Slider: 35 files
  ST - Sweeper: 58 files
Copied 94 files, skipped 0 files.
  CH - Changeup: 1

In [62]:
for i in range(8,16):
    print(i+1)

9
10
11
12
13
14
15
16


In [83]:
# Get file names for each file in the 8-16  most recent games, organized by game date
for i in range(8,16):
    copy_files_to_classification_folders(recent_files_by_game[i], base_dir="modules/naive_classification/val", video_dir = Path("trimmed_woo_pitch_videos"))

Copied 101 files, skipped 0 files.
  CH - Changeup: 3 files
  FF - Fastball: 56 files
  SI - Sinker: 23 files
  SL - Slider: 5 files
  ST - Sweeper: 14 files
Copied 96 files, skipped 0 files.
  CH - Changeup: 6 files
  FF - Fastball: 115 files
  SI - Sinker: 44 files
  SL - Slider: 13 files
  ST - Sweeper: 19 files
Copied 82 files, skipped 0 files.
  CH - Changeup: 9 files
  FF - Fastball: 168 files
  SI - Sinker: 61 files
  SL - Slider: 19 files
  ST - Sweeper: 22 files
Copied 80 files, skipped 0 files.
  CH - Changeup: 18 files
  FF - Fastball: 196 files
  SI - Sinker: 83 files
  SL - Slider: 24 files
  ST - Sweeper: 38 files
Copied 96 files, skipped 0 files.
  CH - Changeup: 22 files
  FF - Fastball: 249 files
  SI - Sinker: 105 files
  SL - Slider: 27 files
  ST - Sweeper: 52 files
Copied 88 files, skipped 0 files.
  CH - Changeup: 27 files
  FF - Fastball: 289 files
  SI - Sinker: 134 files
  SL - Slider: 33 files
  ST - Sweeper: 60 files
Copied 8 files, skipped 0 files.
  CH - Ch

In [66]:
len(recent_files_by_game)

16

In [67]:
len(file_names)

5912

In [68]:
def get_remaining_files(file_names, recent_files_by_game):
    """Return file names that are not in any of the recent_files_by_game lists."""
    recent_set = set()
    for game_files in recent_files_by_game:
        recent_set.update(game_files)
    return [f for f in file_names if f not in recent_set]

remaining_files = get_remaining_files(file_names, recent_files_by_game)
print(f"Total files: {len(file_names)}")
print(f"Recent game files: {sum(len(g) for g in recent_files_by_game)}")
print(f"Remaining files: {len(remaining_files)}")

Total files: 5912
Recent game files: 1242
Remaining files: 4670


In [69]:
remaining_files

['PitchType-CH_Zone-11_PlayID-052f3798-af8f-3cd2-b07d-ea321494c47b_Date-2025-06-17.mp4',
 'PitchType-CH_Zone-11_PlayID-060b0c26-9caf-43d8-b811-2a94bed53c0f_Date-2023-09-18.mp4',
 'PitchType-CH_Zone-11_PlayID-0be2c535-bfab-42d2-a8d4-358e30c67e3a_Date-2024-08-02.mp4',
 'PitchType-CH_Zone-11_PlayID-0e83ceb2-477d-3e53-b6ae-f72acdba6fdf_Date-2025-04-06.mp4',
 'PitchType-CH_Zone-11_PlayID-18fc5860-090b-4a3e-b753-fea9fb841a76_Date-2023-09-18.mp4',
 'PitchType-CH_Zone-11_PlayID-1d59a276-26ce-4136-8999-62d0c40b6d05_Date-2024-09-17.mp4',
 'PitchType-CH_Zone-11_PlayID-2b2d8fd4-c6dc-4c2e-9092-88849e2967d0_Date-2024-05-10.mp4',
 'PitchType-CH_Zone-11_PlayID-3a2074ae-a169-33e9-a00b-ea39c0987728_Date-2025-06-11.mp4',
 'PitchType-CH_Zone-11_PlayID-443565ce-2d2b-32fb-8f71-c778af0e4391_Date-2025-06-28.mp4',
 'PitchType-CH_Zone-11_PlayID-4c9dda04-404a-4960-9179-603a35accf70_Date-2023-08-28.mp4',
 'PitchType-CH_Zone-11_PlayID-51b20068-1138-3d15-a8d8-8c8fc3707145_Date-2025-05-13.mp4',
 'PitchType-CH_Zone-1

In [84]:
for i in range(len(remaining_files)):
    copy_files_to_classification_folders(remaining_files, base_dir="modules/naive_classification/train", video_dir = Path("trimmed_woo_pitch_videos"))

Copied 4670 files, skipped 0 files.
  CH - Changeup: 344 files
  FF - Fastball: 2165 files
  SI - Sinker: 1174 files
  SL - Slider: 567 files
  ST - Sweeper: 420 files
Copied 0 files, skipped 4670 files.
  CH - Changeup: 344 files
  FF - Fastball: 2165 files
  SI - Sinker: 1174 files
  SL - Slider: 567 files
  ST - Sweeper: 420 files
Copied 0 files, skipped 4670 files.
  CH - Changeup: 344 files
  FF - Fastball: 2165 files
  SI - Sinker: 1174 files
  SL - Slider: 567 files
  ST - Sweeper: 420 files
Copied 0 files, skipped 4670 files.
  CH - Changeup: 344 files
  FF - Fastball: 2165 files
  SI - Sinker: 1174 files
  SL - Slider: 567 files
  ST - Sweeper: 420 files
Copied 0 files, skipped 4670 files.
  CH - Changeup: 344 files
  FF - Fastball: 2165 files
  SI - Sinker: 1174 files
  SL - Slider: 567 files
  ST - Sweeper: 420 files
Copied 0 files, skipped 4670 files.
  CH - Changeup: 344 files
  FF - Fastball: 2165 files
  SI - Sinker: 1174 files
  SL - Slider: 567 files
  ST - Sweeper: 42

KeyboardInterrupt: 

In [ ]:
for i in range(len(remaining_files)):
    print(i+1)

In [ ]:
for f in sorted(game_files):
        copy_files_to_classification_folders(f)

copy_files_to_classification_folders(recent_files)